## Masked Autoencoder (MAE) — Training & Evaluation (FD001)

Reconstruction-with-masking baseline:
 - BiLSTM encoder–decoder core with latent bottleneck
 - Random block masking in time & features during training
 - Loss: masked-only or hybrid (masked + α·unmasked) from `config.yaml`
 - Scoring: Monte-Carlo over K random masks (default K=256)
 - Variance-normalization from masked residuals on train normals
 - Validation percentile sweep (F1-optimal threshold)
 - Test-time warm-up calibration per engine (K=50) + run-length filter
 - Saves metrics, scores, and PR curve points to `artifacts/`

## 1) Load Config & Set Seeds
- Read `config.yaml` → `training_mae`, `inference`, `evaluation`
- Enable deterministic ops for reproducibility (when available)


In [10]:
import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

import json, hashlib, random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, f1_score, precision_recall_curve
import yaml

try:
    tf.config.experimental.enable_op_determinism(True)
except Exception:
    pass

def set_seeds(seed=42):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f: json.dump(obj, f, indent=2)
    print("Wrote", path)

ART = Path("artifacts"); ART.mkdir(exist_ok=True, parents=True)
print(f"ART={ART} | TF {tf.__version__} | DET_OPS={os.getenv('TF_DETERMINISTIC_OPS')}")


ART=artifacts | TF 2.10.1 | DET_OPS=1


## 2) Read YAML & Preprocessing Snapshot
 - Enforce critical-field consistency vs `preprocessing_meta.json`
 - Pull knobs for training, inference, and evaluation


In [2]:
cfg_text = Path("config.yaml").read_text(encoding="utf-8")
cfg_hash = hashlib.md5(cfg_text.encode()).hexdigest()
cfg = yaml.safe_load(cfg_text) or {}


meta = json.load((ART/"preprocessing_meta.json").open("r", encoding="utf-8"))

assert int(cfg["data"]["seq_len"]) == int(meta["seq_len"]), "seq_len mismatch vs preprocessing"
assert int(cfg["data"]["stride"])  == int(meta["stride"]),  "stride mismatch vs preprocessing"
if meta.get("yaml_hash") != cfg_hash:
    print("WARN: yaml_hash drift vs preprocessing; proceeding since critical fields match.")


SEED = int(cfg["repro"]["seed"])
set_seeds(SEED)


TMAE = cfg.get("training_mae", {})
MASK_RATIO_TIME = float(TMAE.get("mask_ratio_time", 0.30))
MASK_RATIO_FEAT = float(TMAE.get("mask_ratio_feat", 0.10))
MASK_TOKEN_KIND = str(TMAE.get("mask_token", "zero"))
LOSS_SCOPE      = str(TMAE.get("loss_scope", "hybrid"))
ALPHA_UNMASKED  = float(TMAE.get("hybrid_alpha", 0.05))
LR       = float(TMAE.get("lr", 5e-4))
BATCH    = int(TMAE.get("batch", 64))
EPOCHS   = int(TMAE.get("epochs", 80))
LATENT   = int(TMAE.get("latent", 32))
ENC_UNITS= int(TMAE.get("enc_units", 128))
DEC_UNITS= int(TMAE.get("dec_units", 128))
DROPOUT  = float(TMAE.get("dropout", 0.1))


INF  = cfg.get("inference", {})
K_INFER   = int(INF.get("K_masks", 256))
USE_VNORM = bool(INF.get("variance_normalize", True))

EV  = cfg.get("evaluation", {})
SWP = EV.get("sweep", {}) or {}
pmin = float(SWP.get("pmin", 70.0)); pmax = float(SWP.get("pmax", 99.9)); pstep = float(SWP.get("step", 0.1))
MIN_RUN = int(EV.get("min_run_length", 3))

print("Knobs:",
      f"K={K_INFER} vnorm={USE_VNORM} mrt={MASK_RATIO_TIME} mrf={MASK_RATIO_FEAT}",
      f"token={MASK_TOKEN_KIND} loss={LOSS_SCOPE} alpha={ALPHA_UNMASKED}",
      f"lr={LR} batch={BATCH} epochs={EPOCHS}")


WARN: yaml_hash drift vs preprocessing; proceeding since critical fields match.
Knobs: K=256 vnorm=True mrt=0.3 mrf=0.1 token=zero loss=hybrid alpha=0.05 lr=0.0005 batch=64 epochs=80


## 3) Load Arrays
 - Train normals (`X_train_normal.npy`)
 - Validation windows (`X_val.npy`, `y_val.npy`)


In [3]:
X_tr = np.load(os.path.join(ART,"X_train_normal.npy"))
X_va = np.load(os.path.join(ART,"X_val.npy"))
y_va = np.load(os.path.join(ART,"y_val.npy")).astype(int)

L = int(meta["seq_len"]); D = int(meta["n_features"])
assert X_tr.shape[1] == L and X_va.shape[1] == L
assert X_tr.shape[2] == D and X_va.shape[2] == D
assert len(X_va) == len(y_va)
print(f"X_tr={X_tr.shape} X_va={X_va.shape} y_va={y_va.shape} | L={L} D={D}")


X_tr=(7487, 80, 15) X_va=(2524, 80, 15) y_va=(2524,) | L=80 D=15


## 4) Random Masking Utilities
 - Block masking in time & features per batch
 - Keras `Sequence` that returns ((x, mask), x)
 - Percentile sweep helper (returns best F1 & PR-AUC)


In [4]:
def sample_mask_block_time(batch_size, L, D, mrt=0.3, mrf=0.1, min_block=8):
    k_t = max(min_block, int(round(mrt * L))); k_t = min(k_t, L)
    k_f = max(1, int(round(mrf * D)));       k_f = min(k_f, D)
    t_mask = np.zeros((batch_size, L, 1), dtype=np.float32)
    f_mask = np.zeros((batch_size, 1, D), dtype=np.float32)
    for i in range(batch_size):
        start = np.random.randint(0, L - k_t + 1)
        t_mask[i, start:start+k_t, 0] = 1.0
        idx = np.random.choice(D, size=k_f, replace=False)
        f_mask[i, 0, idx] = 1.0
    return np.clip(t_mask + f_mask, 0, 1)

class RandomMasker(tf.keras.utils.Sequence):
    def __init__(self, X, batch_size=64, mrt=0.3, mrf=0.1, shuffle=True):
        self.X = X; self.bs = batch_size; self.L = X.shape[1]; self.D = X.shape[2]
        self.mrt = float(mrt); self.mrf = float(mrf); self.shuffle = shuffle
        self.idxs = np.arange(len(X))
    def __len__(self): return int(np.ceil(len(self.X) / self.bs))
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.idxs)
    def __getitem__(self, i):
        idx = self.idxs[i*self.bs:(i+1)*self.bs]
        xb = self.X[idx].astype(np.float32)
        m  = sample_mask_block_time(len(xb), self.L, self.D, self.mrt, self.mrf).astype(np.float32)
        return (xb, m), xb

def percentile_sweep(scores, y_true, p_from=70.0, p_to=99.9, step=0.1):
    ps = np.arange(p_from, p_to + 1e-9, step)
    best = {"f1": -1, "p": None, "thr": None, "prec": None, "rec": None}
    for p in ps:
        thr  = np.percentile(scores, p)
        yhat = (scores >= thr).astype(int)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, yhat, average="binary", zero_division=0)
        if f1 > best["f1"]:
            best = {"f1": float(f1), "p": float(p), "thr": float(thr),
                    "prec": float(prec), "rec": float(rec)}
    pr_auc = float(average_precision_score(y_true, scores))
    return best, pr_auc


## 5) Build MAE Core & Trainer
 - Core: BiLSTM encoder (2 layers) → Dense(latent) → LSTM decoder (2 layers)
 - `MAETrainer` wraps masking token and hybrid vs masked-only loss


In [5]:
def build_mae_core(L, D, latent, enc_units, dec_units, dropout):
    x_in = layers.Input(shape=(L, D), name="x_in")
    h = layers.Bidirectional(layers.LSTM(enc_units, return_sequences=True), name="enc_bi_1")(x_in)
    h = layers.Dropout(dropout)(h)
    h = layers.Bidirectional(layers.LSTM(enc_units, return_sequences=False), name="enc_bi_2")(h)
    h = layers.Dropout(dropout)(h)
    z = layers.Dense(latent, activation=None, name="latent")(h)
    d = layers.RepeatVector(L)(z)
    d = layers.LSTM(dec_units, return_sequences=True, name="dec_lstm_1")(d)
    d = layers.Dropout(dropout)(d)
    d = layers.LSTM(dec_units, return_sequences=True, name="dec_lstm_2")(d)
    x_hat = layers.TimeDistributed(layers.Dense(D), name="x_hat")(d)
    return models.Model(x_in, x_hat, name="MAE_BiLSTM_core")

core = build_mae_core(L, D, LATENT, ENC_UNITS, DEC_UNITS, DROPOUT)
print("Core params:", core.count_params())

class MAETrainer(tf.keras.Model):
    def __init__(self, core, feature_dim, alpha_unmasked=0.05, mask_token_kind="zero", loss_scope="hybrid"):
        super().__init__()
        self.core = core
        self.alpha_unmasked = float(alpha_unmasked)
        self.loss_scope = str(loss_scope).lower()
        if mask_token_kind == "learned":
            self.mask_token = self.add_weight("mask_token", shape=(1,1,feature_dim),
                                              initializer="zeros", trainable=True, dtype=tf.float32)
        else:
            self.mask_token = tf.constant(0.0, shape=(1,1,feature_dim), dtype=tf.float32)
        self._loss_tracker = tf.keras.metrics.Mean(name="loss")

    @property
    def metrics(self): return [self._loss_tracker]

    def call(self, inputs, training=False):
        x, m = inputs
        x_eff = x * (1.0 - m) + self.mask_token * m
        return self.core(x_eff, training=training)

    def _loss_masked_only(self, y_true, y_pred, m):
        resid2 = tf.square(y_true - y_pred)
        sum_m  = tf.reduce_sum(resid2 * m, axis=(1,2))
        cnt_m  = tf.maximum(tf.reduce_sum(m, axis=(1,2)), 1.0)
        return tf.reduce_mean(sum_m / cnt_m)

    def _loss_hybrid(self, y_true, y_pred, m):
        resid2 = tf.square(y_true - y_pred)
        sum_m  = tf.reduce_sum(resid2 * m, axis=(1,2))
        cnt_m  = tf.maximum(tf.reduce_sum(m, axis=(1,2)), 1.0)
        um     = 1.0 - m
        sum_um = tf.reduce_sum(resid2 * um, axis=(1,2))
        cnt_um = tf.maximum(tf.reduce_sum(um, axis=(1,2)), 1.0)
        return tf.reduce_mean((sum_m / cnt_m) + self.alpha_unmasked * (sum_um / cnt_um))

    def train_step(self, data):
        (x, m), y = data
        with tf.GradientTape() as tape:
            y_pred = self((x, m), training=True)
            loss = self._loss_masked_only(y, y_pred, m) if self.loss_scope in ("masked_only","masked") else self._loss_hybrid(y, y_pred, m)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self._loss_tracker.update_state(loss)
        return {"loss": self._loss_tracker.result()}

    def test_step(self, data):
        (x, m), y = data
        y_pred = self((x, m), training=False)
        loss = self._loss_masked_only(y, y_pred, m) if self.loss_scope in ("masked_only","masked") else self._loss_hybrid(y, y_pred, m)
        self._loss_tracker.update_state(loss)
        return {"loss": self._loss_tracker.result()}

mae = MAETrainer(core, feature_dim=D,
                 alpha_unmasked=ALPHA_UNMASKED,
                 mask_token_kind=MASK_TOKEN_KIND,
                 loss_scope=LOSS_SCOPE)
mae.compile(optimizer=keras.optimizers.Adam(LR))
print(f"[Trainer] token={MASK_TOKEN_KIND} loss={LOSS_SCOPE} alpha={ALPHA_UNMASKED} lr={LR}")


Core params: 765871
[Trainer] token=zero loss=hybrid alpha=0.05 lr=0.0005


## 6) Train with Random Masking
 - Split 10% of train normals as internal validation for early stopping
 - Callbacks: ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
 - Saves: `model_mae_core.h5`, `train_log_mae.json`, optional `mask_token.npy`


In [6]:
N = len(X_tr)
iv_take = max(1, int(round(0.10 * N)))
idx = np.arange(N); np.random.seed(SEED); np.random.shuffle(idx)
X_tr_i, X_iv_i = X_tr[idx[iv_take:]], X_tr[idx[:iv_take]]

train_seq = RandomMasker(X_tr_i, batch_size=BATCH, mrt=MASK_RATIO_TIME, mrf=MASK_RATIO_FEAT, shuffle=True)
ival_seq  = RandomMasker(X_iv_i, batch_size=BATCH, mrt=MASK_RATIO_TIME, mrf=MASK_RATIO_FEAT, shuffle=False)

ckpt = keras.callbacks.ModelCheckpoint(ART/"ckpt_mae_val_loss.h5",
                                       monitor="val_loss", save_best_only=True, save_weights_only=True)
es   = keras.callbacks.EarlyStopping(monitor="val_loss",
                                     patience=int(TMAE.get("early_stopping_patience",10)),
                                     min_delta=float(TMAE.get("early_stopping_min_delta",1e-4)),
                                     restore_best_weights=True)
rlr  = keras.callbacks.ReduceLROnPlateau(monitor="val_loss",
                                         patience=int(TMAE.get("rlr_patience",5)),
                                         factor=0.5, min_lr=1e-6)

hist = mae.fit(train_seq, validation_data=ival_seq, epochs=EPOCHS, callbacks=[ckpt, es, rlr], verbose=1)

core.save(ART/"model_mae_core.h5")
save_json({"history": {k:[float(x) for x in v] for k,v in hist.history.items()},
           "params": int(core.count_params()),
           "yaml_hash": cfg_hash,
           "config_snapshot": {"training_mae": TMAE, "inference": INF, "evaluation": EV}},
          ART/"train_log_mae.json")
if isinstance(getattr(mae,"mask_token",None), tf.Variable):
    np.save(ART/"mask_token.npy", mae.mask_token.numpy())
print("Saved model + training log.")


Epoch 1/80
106/106 [==============================] - 13s 75ms/step - loss: 0.5178 - val_loss: 0.4800 - lr: 5.0000e-04
Epoch 2/80
106/106 [==============================] - 6s 61ms/step - loss: 0.4623 - val_loss: 0.4614 - lr: 5.0000e-04
Epoch 3/80
106/106 [==============================] - 6s 60ms/step - loss: 0.4536 - val_loss: 0.4495 - lr: 5.0000e-04
Epoch 4/80
106/106 [==============================] - 6s 61ms/step - loss: 0.4475 - val_loss: 0.4472 - lr: 5.0000e-04
Epoch 5/80
106/106 [==============================] - 6s 60ms/step - loss: 0.4450 - val_loss: 0.4399 - lr: 5.0000e-04
Epoch 6/80
106/106 [==============================] - 6s 60ms/step - loss: 0.4391 - val_loss: 0.4434 - lr: 5.0000e-04
Epoch 7/80
106/106 [==============================] - 6s 59ms/step - loss: 0.4400 - val_loss: 0.4332 - lr: 5.0000e-04
Epoch 8/80
106/106 [==============================] - 6s 58ms/step - loss: 0.4387 - val_loss: 0.4442 - lr: 5.0000e-04
Epoch 9/80
106/106 [==============================] - 6

## 7) Validation Scoring & Threshold Sweep
 - Estimate variance vector from **masked residuals** on train normals
 - Score validation with K random masks (Monte-Carlo)
 - Sweep percentiles for F1-optimal threshold
 - Saves: `feat_var_vnorm_mae.npy`, `scores_mae_val.npy`, `metrics_mae_val.json`, `pr_mae_val.npy`


In [7]:
def load_mask_token_from_artifacts(art_dir, token_kind, trainer_obj=None):
    if token_kind != "learned": return None
    if trainer_obj is not None and isinstance(getattr(trainer_obj, "mask_token", None), tf.Variable):
        return trainer_obj.mask_token.numpy()
    p = Path(art_dir)/"mask_token.npy"
    return np.load(p) if p.exists() else None

def masked_residual_var_core(core_model, X, K, mrt, mrf, token_kind="zero", mask_token_var=None, bs=128):
    D = X.shape[2]
    res_sq = np.zeros(D, dtype=np.float64); cnt = np.zeros(D, dtype=np.float64)
    for _ in range(K):
        for i in range(0, len(X), bs):
            xb = X[i:i+bs].astype(np.float32)
            m  = sample_mask_block_time(len(xb), xb.shape[1], xb.shape[2], mrt, mrf).astype(np.float32)
            x_eff = xb * (1.0 - m)
            if token_kind == "learned" and mask_token_var is not None:
                x_eff = x_eff + mask_token_var * m
            xh = core_model.predict_on_batch(x_eff)
            resid = (xb - xh) * m
            res_sq += np.sum(resid**2, axis=(0,1)); cnt += np.sum(m, axis=(0,1))
    return (res_sq / np.maximum(cnt, 1.0)).astype(np.float32)

def mae_scores_core(core_model, X, K, mrt, mrf, vnorm=None, token_kind="zero", mask_token_var=None, bs=128):
    scores = np.zeros(len(X), dtype=np.float32)
    denom_vec = None
    if vnorm is not None:
        denom_vec = np.clip(vnorm, 1e-8, None).astype(np.float32)[None, None, :]
    for i in range(0, len(X), bs):
        xb = X[i:i+bs].astype(np.float32)
        acc = np.zeros(len(xb), dtype=np.float64)
        for _ in range(K):
            m  = sample_mask_block_time(len(xb), xb.shape[1], xb.shape[2], mrt, mrf).astype(np.float32)
            x_eff = xb * (1.0 - m)
            if token_kind == "learned" and mask_token_var is not None:
                x_eff = x_eff + mask_token_var * m
            xh = core_model.predict_on_batch(x_eff)
            resid2 = ((xb - xh)**2) * m
            if denom_vec is not None: resid2 = resid2 / denom_vec
            denom_count = np.maximum(np.sum(m, axis=(1,2)), 1.0)
            acc += np.sum(resid2, axis=(1,2)) / denom_count
        scores[i:i+bs] = (acc / K).astype(np.float32)
    return scores

mask_token_np = load_mask_token_from_artifacts(ART, MASK_TOKEN_KIND, trainer_obj=mae)
K_for_var = max(256, K_INFER)
feat_var = masked_residual_var_core(core, X_tr, K=K_for_var, mrt=MASK_RATIO_TIME, mrf=MASK_RATIO_FEAT,
                                    token_kind=MASK_TOKEN_KIND, mask_token_var=mask_token_np)
np.save(ART/"feat_var_vnorm_mae.npy", feat_var)

vnorm_vec = feat_var if USE_VNORM else None
s_va = mae_scores_core(core, X_va, K=K_INFER, mrt=MASK_RATIO_TIME, mrf=MASK_RATIO_FEAT,
                       vnorm=vnorm_vec, token_kind=MASK_TOKEN_KIND, mask_token_var=mask_token_np)

best, pr_auc = percentile_sweep(s_va, y_va, p_from=pmin, p_to=pmax, step=pstep)
metrics_val = {"precision": best["prec"], "recall": best["rec"], "f1": best["f1"],
               "pr_auc": pr_auc, "best_percentile": best["p"], "threshold": float(best["thr"]),
               "scoring": {"type": "masked_mse_varnorm" if USE_VNORM else "masked_mse_raw",
                           "K": K_INFER, "mask_ratio_time": MASK_RATIO_TIME, "mask_ratio_feat": MASK_RATIO_FEAT,
                           "mask_token": MASK_TOKEN_KIND, "loss_scope": LOSS_SCOPE, "alpha_unmasked": ALPHA_UNMASKED},
               "params": int(core.count_params()), "yaml_hash": cfg_hash}
print("[VAL]", metrics_val)
save_json(metrics_val, ART/"metrics_mae_val.json")
np.save(ART/"scores_mae_val.npy", s_va)


[VAL] {'precision': 0.5428109854604201, 'recall': 0.49411764705882355, 'f1': 0.5173210161662818, 'pr_auc': 0.6122205615538494, 'best_percentile': 75.49999999999969, 'threshold': 1.1304305428266486, 'scoring': {'type': 'masked_mse_varnorm', 'K': 256, 'mask_ratio_time': 0.3, 'mask_ratio_feat': 0.1, 'mask_token': 'zero', 'loss_scope': 'hybrid', 'alpha_unmasked': 0.05}, 'params': 765871, 'yaml_hash': 'fff0c383d19eb2a711443c4ba712b508'}
Wrote artifacts\metrics_mae_val.json


## 8) Test Scoring (Raw) & Metrics
 - Reuse validation threshold
 - Save: `scores_mae_test.npy`, `metrics_mae_test.json`, `pr_mae_test.npy`


In [8]:
X_te = np.load(ART/"X_test.npy")
y_te = np.load(ART/"y_test.npy").astype(int)
print("[TEST] shapes:", X_te.shape, y_te.shape, "| positives:", int(y_te.sum()))

valm = json.load((ART/"metrics_mae_val.json").open("r"))
thr    = float(valm["threshold"])
K      = int(valm["scoring"]["K"])
mrt    = float(valm["scoring"]["mask_ratio_time"])
mrf    = float(valm["scoring"]["mask_ratio_feat"])
token  = valm["scoring"]["mask_token"]
use_v  = (valm["scoring"]["type"] == "masked_mse_varnorm")
vnorm_vec = np.load(ART/"feat_var_vnorm_mae.npy") if use_v else None
mask_token_np = np.load(ART/"mask_token.npy") if (token=="learned" and (ART/"mask_token.npy").exists()) else None

s_te = mae_scores_core(core, X_te, K=K, mrt=mrt, mrf=mrf, vnorm=vnorm_vec,
                       token_kind=token, mask_token_var=mask_token_np)
np.save(ART/"scores_mae_test.npy", s_te)

yhat = (s_te >= thr).astype(int)
prec, rec, f1, _ = precision_recall_fscore_support(y_te, yhat, average="binary", zero_division=0)
pr_auc = float(average_precision_score(y_te, s_te))

out = {"precision": float(prec), "recall": float(rec), "f1": float(f1), "pr_auc": pr_auc,
       "threshold": thr, "best_percentile_from_val": float(valm["best_percentile"]),
       "n_windows": int(len(X_te)), "n_pos": int(y_te.sum()), "scoring": valm["scoring"]}
print("[TEST MAE]", out)
save_json(out, ART/"metrics_mae_test.json")


[TEST] shapes: (5660, 80, 15) (5660,) | positives: 407
[TEST MAE] {'precision': 0.08488612836438923, 'recall': 0.3022113022113022, 'f1': 0.13254310344827586, 'pr_auc': 0.10852788422075473, 'threshold': 1.1304305428266486, 'best_percentile_from_val': 75.49999999999969, 'n_windows': 5660, 'n_pos': 407, 'scoring': {'type': 'masked_mse_varnorm', 'K': 256, 'mask_ratio_time': 0.3, 'mask_ratio_feat': 0.1, 'mask_token': 'zero', 'loss_scope': 'hybrid', 'alpha_unmasked': 0.05}}
Wrote artifacts\metrics_mae_test.json


## 9) Warm-Up Calibration & PR Curve (Test)
 - Per-engine z-score using first K=50 windows
 - Threshold picked on warm-up VAL, then applied to TEST
 - Saves: `scores_mae_val_warmup.npy`, `scores_mae_test_warmup.npy`, `pr_mae_test_warmup.npy`, `metrics_mae_calibrated.json`


In [11]:
E = cfg.get("evaluation", {})
K_WARM  = int(E.get("calibration",{}).get("warmup",{}).get("K", 50))
MIN_RUN = int(E.get("min_run_length", 3))

def warmup_zscore(scores, unit_ids, K=50):
    out = scores.astype(np.float32, copy=True)
    for uid in np.unique(unit_ids):
        idx = np.where(unit_ids == uid)[0]
        base = out[idx[:min(K, len(idx))]]
        mu, sd = float(base.mean()), float(base.std())
        if sd < 1e-8: sd = 1.0
        out[idx] = (out[idx] - mu) / sd
    return out

def run_length_filter(pred_bin, min_run=1):
    if min_run <= 1: return pred_bin.astype(int)
    y = pred_bin.astype(int).copy(); i=0
    while i < len(y):
        if y[i]==1:
            j=i
            while j < len(y) and y[j]==1: j+=1
            if (j-i) < min_run: y[i:j]=0
            i=j
        else: i+=1
    return y

# load raw scores + ids
s_val = np.load(ART/"scores_mae_val.npy");  y_val = np.load(ART/"y_val.npy").astype(int);  ids_val = np.load(ART/"unit_ids_val.npy")
s_tst = np.load(ART/"scores_mae_test.npy"); y_tst = np.load(ART/"y_test.npy").astype(int); ids_tst = np.load(ART/"unit_ids_test.npy")

# warm-up z-score per engine
s_val_w = warmup_zscore(s_val, ids_val, K=K_WARM)
s_tst_w = warmup_zscore(s_tst, ids_tst, K=K_WARM)
np.save(ART/"scores_mae_val_warmup.npy", s_val_w)
np.save(ART/"scores_mae_test_warmup.npy", s_tst_w)

# select percentile on VAL (warm-up)
best = {"f1": -1.0}
for p in np.arange(pmin, pmax + 1e-9, pstep):
    thr = np.percentile(s_val_w, p)
    yhat = run_length_filter((s_val_w >= thr).astype(int), MIN_RUN)
    f1 = f1_score(y_val, yhat, zero_division=0)
    if f1 > best["f1"]:
        best = {"p": float(p), "thr": float(thr), "f1": float(f1)}

# apply fixed warm-up threshold to TEST
yhat_t = run_length_filter((s_tst_w >= best["thr"]).astype(int), MIN_RUN)
pr, rc, f1, _ = precision_recall_fscore_support(y_tst, yhat_t, average="binary", zero_division=0)
ap_raw  = float(average_precision_score(y_tst, s_tst))
ap_warm = float(average_precision_score(y_tst, s_tst_w))
print(f"[MAE] PR-AUC raw={ap_raw:.3f}  warm-up={ap_warm:.3f}  | TEST F1={f1:.3f} P={pr:.3f} R={rc:.3f} @pct≈{best['p']:.1f}")

# PR curve for warm-up
prec_w, rec_w, thr_w = precision_recall_curve(y_tst, s_tst_w)
np.save(ART/"pr_mae_test_warmup.npy", np.c_[rec_w[:-1], prec_w[:-1], thr_w])

payload = {
    "raw":  {"val_pr_auc": float(average_precision_score(y_val, s_val)),
             "test_pr_auc": ap_raw},
    "warm": {"val_pr_auc": float(average_precision_score(y_val, s_val_w)),
             "test_pr_auc": ap_warm,
             "threshold": best["thr"], "percentile": best["p"],
             "postproc": {"min_run_length": MIN_RUN},
             "test_confusion_at_thresh": {"precision": float(pr), "recall": float(rc), "f1": float(f1)}},
    "calibration": {"warmup": {"enabled": True, "K": K_WARM}}
}
save_json(payload, ART/"metrics_mae_calibrated.json")


[MAE] PR-AUC raw=0.109  warm-up=0.187  | TEST F1=0.183 P=0.242 R=0.147 @pct≈79.8
Wrote artifacts\metrics_mae_calibrated.json
